# MA Crossover Strategy Backtest

> Strategy ID: MA_CROSSOVER_V1 | Based on: ma_crossover_strategy_spec.md
> Type: Trend Following | Market: A-Shares (Long Only)
> Target: 603986.SH

## Pipeline

1. Load Data -> 2. Calculate MA -> 3. Generate Signals -> 4. Run Backtest -> 5. Metrics -> 6. Visualize

## Switch Stocks

Change the CHOSEN variable in Cell 2. Available: 002594.SZ, 603986.SH, 688981.SH

In [ ]:
# ============================================================
# 1. Parameters (per spec section 2)
# ============================================================
FAST_PERIOD   = 10
SLOW_PERIOD   = 60
MA_TYPE       = "ema"
INITIAL_CAPITAL = 1_000_000
COMMISSION_RATE = 0.0003
STAMP_TAX_RATE  = 0.0005
SLIPPAGE_RATE   = 0.001
print(f"Strategy: {MA_TYPE.upper()}({FAST_PERIOD}) / {MA_TYPE.upper()}({SLOW_PERIOD})")
print(f"Costs: comm={COMMISSION_RATE:.3%} stamp={STAMP_TAX_RATE:.3%}(sell) slip={SLIPPAGE_RATE:.1%}")
print(f"Capital: {INITIAL_CAPITAL:,}")

In [ ]:
# ============================================================
# 2. Load Data
# ============================================================
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings('ignore')

STOCKS = {
    "002594.SZ": "C:/Users/RYAN/Desktop/BA工作坊/data/002594.SZ_比亚迪/daily_adjusted.csv",
    "603986.SH": "C:/Users/RYAN/Desktop/BA工作坊/data/603986.SH_兆易创新/daily_adjusted.csv",
    "688981.SH": "C:/Users/RYAN/Desktop/BA工作坊/data/688981.SH_中芯国际/daily_adjusted.csv",
}

# Change this to switch stocks (e.g. "002594.SZ" or "688981.SH")
CHOSEN = "603986.SH"

df = pd.read_csv(STOCKS[CHOSEN], encoding="utf-8-sig", parse_dates=["trade_date"])
df = df.sort_values("trade_date").reset_index(drop=True)
df["price"] = df["close_qfq"]
print(f"Stock: {CHOSEN}")
print(f"Rows: {len(df)}, {df.trade_date.min().date()} ~ {df.trade_date.max().date()}")
print(f"Price range: {df.price.min():.2f} ~ {df.price.max():.2f}")
df[["trade_date","price","vol","amount"]].head()

In [ ]:
# ============================================================
# 3. Moving Averages (per spec section 2)
# ============================================================
def calc_ma(s, n, t):
    return s.ewm(span=n, adjust=False).mean() if t == "ema" else s.rolling(n).mean()

df["ma_fast"] = calc_ma(df["price"], FAST_PERIOD, MA_TYPE)
df["ma_slow"] = calc_ma(df["price"], SLOW_PERIOD, MA_TYPE)
warmup = max(FAST_PERIOD, SLOW_PERIOD)
df_v = df.iloc[warmup:].copy().reset_index(drop=True)
fast_lbl = f"{MA_TYPE.upper()}({FAST_PERIOD})"
slow_lbl = f"{MA_TYPE.upper()}({SLOW_PERIOD})"
print(f"Warmup bars dropped: {warmup}")
print(f"Valid rows: {len(df_v)}")
df_v[["trade_date","price","ma_fast","ma_slow"]].head(10).round(2)

In [ ]:
# ============================================================
# 4. Signal Generation
# ============================================================
df_v["signal"]   = (df_v["ma_fast"] > df_v["ma_slow"]).astype(int)
df_v["cross"]    = df_v["signal"].diff()
df_v["position"] = df_v["signal"].shift(1).fillna(0).astype(int)
n_g = int((df_v["cross"] == 1).sum())
n_d = int((df_v["cross"] == -1).sum())
print(f"Golden Cross: {n_g} | Death Cross: {n_d} | Pairs: {n_g}")
changes = df_v[df_v["cross"] != 0].copy()
changes["label"] = changes["cross"].map({1: "GOLDEN BUY", -1: "DEATH SELL"})
changes[["trade_date","price","label"]].head(10)

In [ ]:
# ============================================================
# 5. Backtest (with all costs)
# ============================================================
df_v["price_ret"]  = df_v["price"].pct_change()
df_v["gross_ret"]  = df_v["position"] * df_v["price_ret"]
df_v["trade_act"]  = df_v["position"].diff().abs()
df_v["cost"]       = df_v["trade_act"] * (COMMISSION_RATE + SLIPPAGE_RATE)
df_v.loc[df_v["position"].diff() < 0, "cost"] += STAMP_TAX_RATE
df_v["strat_ret"]  = df_v["gross_ret"] - df_v["cost"]
df_v["equity"]     = (1 + df_v["strat_ret"].fillna(0)).cumprod()
df_v["bench_eq"]   = (1 + df_v["price_ret"].fillna(0)).cumprod()
pk_s = df_v["equity"].cummax(); pk_b = df_v["bench_eq"].cummax()
df_v["dd"]       = (df_v["equity"] - pk_s) / pk_s
df_v["bench_dd"] = (df_v["bench_eq"] - pk_b) / pk_b
print(f"Strategy equity: {df_v.equity.iloc[-1]:.4f} ({df_v.equity.iloc[-1]-1:+.2%})")
print(f"Benchmark eq:    {df_v.bench_eq.iloc[-1]:.4f} ({df_v.bench_eq.iloc[-1]-1:+.2%})")

In [ ]:
# ============================================================
# 6. Performance Metrics
# ============================================================
ret = df_v["strat_ret"].dropna()
N = len(ret)
total_ret = df_v["equity"].iloc[-1] - 1
bench_ret = df_v["bench_eq"].iloc[-1] - 1
ann_ret   = (1+total_ret)**(252/N)-1
ann_bench = (1+bench_ret)**(252/N)-1
rc = ret[ret!=0] if len(ret[ret!=0])>0 else ret
sharpe = np.sqrt(252)*rc.mean()/rc.std() if rc.std()>0 else 0
max_dd = df_v["dd"].min()
max_dd_b = df_v["bench_dd"].min()
calmar = ann_ret/abs(max_dd) if max_dd!=0 else 0

g_entry = df_v[df_v["cross"]==1]
trades_ret = []
for i in range(len(g_entry)):
    ei = df_v[df_v["trade_date"]==g_entry["trade_date"].iloc[i]].index[0]
    xi = len(df_v)-1 if i+1>=len(g_entry) else df_v[df_v["trade_date"]==g_entry["trade_date"].iloc[i+1]].index[0]-1
    trades_ret.append(df_v.loc[xi,"price"]/df_v.loc[ei,"price"]-1)
wins=[r for r in trades_ret if r>0]; loses=[r for r in trades_ret if r<=0]
wr=len(wins)/len(trades_ret) if trades_ret else 0
aw=np.mean(wins) if wins else 0
al=np.mean([abs(r) for r in loses]) if loses else 0
plr=aw/al if al>0 else float("inf")

from IPython.display import display, HTML
display(HTML(f"""<table style="border-collapse:collapse;width:100%;font-size:14px">
<tr style="background:#f8f9fa;font-weight:bold"><td style="padding:10px 16px">Metric</td><td style="padding:10px 16px;text-align:right">Strategy</td><td style="padding:10px 16px;text-align:right">Benchmark</td></tr>
<tr><td>Total Return</td><td style="text-align:right;color:{'#D85A30' if total_ret>0 else '#1D9E75'}">{total_ret:+.2%}</td><td style="text-align:right;color:{'#D85A30' if bench_ret>0 else '#1D9E75'}">{bench_ret:+.2%}</td></tr>
<tr style="background:#f8f9fa"><td>Annual Return</td><td style="text-align:right">{ann_ret:+.2%}</td><td style="text-align:right">{ann_bench:+.2%}</td></tr>
<tr><td>Sharpe (ann)</td><td style="text-align:right;font-weight:bold">{sharpe:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Max Drawdown</td><td style="text-align:right;color:#D85A30">{max_dd:+.2%}</td><td style="text-align:right;color:#D85A30">{max_dd_b:+.2%}</td></tr>
<tr><td>Calmar Ratio</td><td style="text-align:right;font-weight:bold">{calmar:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Win Rate</td><td style="text-align:right">{wr:.1%}</td><td style="text-align:right">-</td></tr>
<tr><td>P/L Ratio</td><td style="text-align:right;font-weight:bold">{plr:.2f}</td><td style="text-align:right">-</td></tr>
<tr style="background:#f8f9fa"><td>Total Trades</td><td style="text-align:right">{len(trades_ret)}</td><td style="text-align:right">-</td></tr>
</table>"""))

In [ ]:
# ============================================================
# 7. Visualization
# ============================================================
import matplotlib.pyplot as plt, matplotlib.dates as mdates
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(3,1,figsize=(16,12),gridspec_kw={"height_ratios":[2,1.5,1]})
dts = df_v["trade_date"]

ax=axes[0]
ax.plot(dts,df_v["price"],"#B4B2A9",lw=0.8,alpha=0.7,label="Close")
ax.plot(dts,df_v["ma_fast"],"#D85A30",lw=1.2,label=fast_lbl)
ax.plot(dts,df_v["ma_slow"],"#534AB7",lw=1.2,label=slow_lbl)
g=df_v[df_v["cross"]==1]; d=df_v[df_v["cross"]==-1]
ax.scatter(g["trade_date"],g["price"],c="#1D9E75",marker="^",s=80,zorder=5,label="Golden")
ax.scatter(d["trade_date"],d["price"],c="#D85A30",marker="v",s=80,zorder=5,label="Death")
ax.set_title(f"{CHOSEN} - Price & MA & Signals",fontsize=14,fontweight="bold")
ax.legend(loc="upper left",fontsize=9); ax.set_ylabel("Price"); ax.grid(True,alpha=0.3)

ax=axes[1]
ax.fill_between(dts,1,df_v["equity"],alpha=0.1,color="#378ADD")
ax.plot(dts,df_v["equity"],"#378ADD",lw=1.8,label="Strategy")
ax.plot(dts,df_v["bench_eq"],"#B4B2A9",lw=1.2,ls="dashed",label="Benchmark")
ax.axhline(1,c="black",lw=0.5,ls="dotted")
ax.set_title("Equity Curve",fontsize=14,fontweight="bold")
ax.legend(loc="upper left",fontsize=9); ax.set_ylabel("Equity"); ax.grid(True,alpha=0.3)

ax=axes[2]
ax.fill_between(dts,df_v["dd"],0,alpha=0.3,color="#D85A30")
ax.fill_between(dts,df_v["bench_dd"],0,alpha=0.15,color="#B4B2A9")
ax.plot(dts,df_v["dd"],"#D85A30",lw=1,label="Strategy DD")
ax.plot(dts,df_v["bench_dd"],"#B4B2A9",lw=0.8,ls="dashed",label="Benchmark DD")
ax.axhline(0,c="black",lw=0.3); ax.axhline(-0.25,c="#D85A30",lw=0.5,ls="dotted",alpha=0.5)
ax.set_title("Drawdown",fontsize=14,fontweight="bold")
ax.legend(loc="lower left",fontsize=9); ax.set_ylabel("Drawdown"); ax.grid(True,alpha=0.3)
for a in axes:
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    a.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(a.xaxis.get_majorticklabels(),rotation=45,ha="right",fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. Summary
# ============================================================
print(f"""
{'='*60}
  MA Crossover Backtest: {CHOSEN}
  {MA_TYPE.upper()}({FAST_PERIOD}) / {MA_TYPE.upper()}({SLOW_PERIOD})
  Return: {total_ret:+.2%} | Sharpe: {sharpe:.2f} | MaxDD: {max_dd:+.2%}
  WinRate: {wr:.1%} | Trades: {len(trades_ret)} | P/L Ratio: {plr:.2f}
{'='*60}

To switch stocks: change CHOSEN in Cell 2
To tune params: FAST_PERIOD / SLOW_PERIOD / MA_TYPE
Based on ma_crossover_strategy_spec.md
""")